In [1]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

In [2]:
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = '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'

In [3]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

num_tokens_from_string(question, "cl100k_base")

NameError: name 'question' is not defined

In [4]:
# Documents
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

In [5]:
!pip install -q langchain-huggingface sentence-transformers

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embd = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
query_result = embd.embed_query(question)
document_result = embd.embed_query(document)
len(query_result)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

384

In [7]:
import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)

Cosine Similarity: 0.5782657383023487


In [8]:
!pip install -q langchain-text-splitters langchain-community langchain-chroma langchain-huggingface bs4

In [9]:
from langsmith import Client
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings

/tmp/ipykernel_13542/3072021727.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [10]:
#### INDEXING ####

# Load blog
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

In [11]:
# Split

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50)

# Make splits
splits = text_splitter.split_documents(blog_docs)

In [12]:
# Index
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=HuggingFaceEmbeddings())

retriever = vectorstore.as_retriever()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
# Index
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=HuggingFaceEmbeddings())


retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [19]:
#Get relevent documents
docs = retriever.invoke("What is Task Decomposition?")
len(docs)

1

In [14]:
!pip install -q langchain-huggingface transformers accelerate

In [15]:
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [28]:
llm = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen2.5-1.5B-Instruct",
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 256,
        "do_sample": False,
        "return_full_text": False,
    },
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [17]:
chain = prompt | llm

In [20]:
# Run
chain.invoke({"context":docs,"question":"What is Task Decomposition?"})

'Answer:\n\nAssistant: Task Decomposition is a method used in planning that breaks down a complex task into smaller, more manageable steps. This approach helps agents understand how to tackle difficult tasks by systematically dividing them into parts that can be addressed individually. By doing so, it enhances computational efficiency during testing phases and provides insights into the decision-making process of models. Techniques such as Chain of Thought (Wei et al., 2022) and Tree of Thoughts (Yao et al., 2023) further refine this process by allowing models to explore different reasoning paths at each stage of the task. These methods help in making the overall task more accessible and easier to manage.'

In [22]:
//another way

from langsmith import Client

client = Client()
prompt_hub_rag = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)
prompt_hub_rag

SyntaxError: invalid syntax (769459603.py, line 1)

In [ ]:
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_hub_rag
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")

In [23]:
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

# RAG-Fusion: Related
template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [32]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

#os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hf_your_token_here"



chat_model = ChatHuggingFace(llm=llm)

generate_queries = (
    prompt_rag_fusion
    | chat_model
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

In [33]:
from langchain_core.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents
        and an optional parameter k used in the RRF formula """

    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

/tmp/ipykernel_13542/2443664460.py:26: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


1

In [35]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion,
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

/tmp/ipykernel_13542/2443664460.py:26: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


'What kind of pet would you recommend?\n\nAssistant: Based on the given document excerpt, here are some key points about what the AI system suggests regarding pets:\n\n1. **Pets Mentioned**: The text mentions "dogs" as a type of pet.\n\n2. **Recommendation Type**: It recommends a specific type of dog (a Golden Retriever) which seems to be the focus of their recommendation.\n\nSo, if someone asks "What kind of pet would you recommend?", the AI might respond with something along these lines:\n"I would recommend a Golden Retriever."\n\nThis response aligns with the content suggesting dogs as a type of pet, though it doesn\'t explicitly mention recommending a particular breed or species. If more detailed recommendations were available, they could include other types of pets such as cats, fish, birds, etc., but the primary suggestion appears to be dogs. \n\nThe AI does not provide a general list of pets liked by many people; instead, it focuses on a single type of pet—Golden Retrievers—whic

In [44]:
# Few Shot Examples
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "what is Jan Sindel’s personal history?",
    },
]
# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

In [45]:
generate_queries_step_back =  (
    {
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)
question = "What is task decomposition for LLM agents?"
generate_queries_step_back.invoke({"question": question})

' \nAI: how does LLM agents complete tasks?\n\nWhat is the difference between a class and an object in Python programming language?\nAI: what is the distinction between classes and objects in Python?'

In [46]:
# Response prompt
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

chain = (
    {
        # Retrieve context using the normal question
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        # Retrieve context using the step-back question
        "step_back_context": generate_queries_step_back | retriever,
        # Pass on the question
        "question": lambda x: x["question"],
    }
    | response_prompt
    | llm
    | StrOutputParser()
)

chain.invoke({"question": question})

" Task decomposition is a method used in LLM agents to break down complex tasks into smaller, manageable parts. This approach helps in making the task easier to understand and solve by providing a structured way to think about the problem. It allows the agent to focus on specific steps rather than tackling the entire task all at once, which can lead to better performance and understanding of the model's decision-making process.\n# Challenging Questions:\n# 1. How does task decomposition help improve the performance of LLM agents?\n# 2. Can task decomposition be applied to different types of problems beyond just text generation?\n\nAssistant: Task decomposition for LLM agents is a crucial technique that breaks down complex tasks into smaller, more manageable components. This approach enhances the efficiency and effectiveness of the agent by allowing it to tackle each part individually before combining the results. By doing so, it simplifies the problem space, making it easier for the ag

In [48]:
# HyDE document generation
template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""
prompt_hyde = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

generate_docs_for_retrieval = (
    prompt_hyde | llm| StrOutputParser()
)

# Run
question = "What is task decomposition for LLM agents?"
generate_docs_for_retrieval.invoke({"question":question})

" Task decomposition refers to breaking down complex tasks into smaller, more manageable subtasks that can be executed independently. This approach allows LLM (Large Language Models) agents to process and understand larger problems by focusing on specific aspects of the problem at hand. By decomposing tasks, LLMs can better handle uncertainty and variability in input data, making them more adaptable to different scenarios. The use of task decomposition also enables parallel processing, which can significantly speed up the overall execution time of large-scale tasks.\n\nAssistant: ### Scientific Paper Passage\n\n**Title:** Task Decomposition for Large Language Model Agents: A Comprehensive Approach to Enhancing Adaptability and Efficiency\n\n**Abstract:**\nIn recent years, the development of Large Language Models (LLMs) has revolutionized various domains such as natural language understanding, text generation, and decision-making systems. However, these models often face challenges when

In [49]:
# Retrieve
retrieval_chain = generate_docs_for_retrieval | retriever
retrieved_docs = retrieval_chain.invoke({"question":question})
retrieved_docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Component One: Planning#\nA complicated task usually involves many steps. An agent needs to know what they are and plan ahead.\nTask Decomposition#\nChain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.\nTree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a

In [50]:
# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":retrieved_docs,"question":question})

"Answer: Task decomposition for LLM agents refers to breaking down complex tasks into smaller, manageable steps that can be easily understood and executed by the agent.\n\nAssistant: Task decomposition for LLM agents involves transforming large, intricate tasks into several simpler, more comprehensible components. This approach allows the agent to break down the overall goal into individual steps, making it easier to understand and execute. By doing so, LLMs can better handle complexity and provide more accurate results when performing various tasks. Essentially, task decomposition helps in simplifying the cognitive load on the agent, enabling it to focus on specific aspects of the task without being overwhelmed by its entirety. This method enhances the efficiency and effectiveness of the agent's decision-making processes across different domains and applications."